# Deep Learning Training: Breast Histopathology Cancer Detection

This notebook trains a custom CNN for binary classification of histopathology images.

## Approach
- Custom CNN architecture optimized for 50x50 patches
- Binary classification (IDC+ vs No Cancer)
- Data augmentation for class imbalance handling
- Batch normalization and dropout for regularization

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import sys
sys.path.append('../src')

# Import Kaggle downloader
try:
    from kaggle_downloader import setup_project_data, check_kaggle_credentials
    KAGGLE_AVAILABLE = True
except ImportError:
    KAGGLE_AVAILABLE = False
    print("⚠️ Kaggle downloader not available - install kaggle package")


from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')

## 1. Data Preparation

In [ ]:
# Load and preprocess images
import cv2
from tensorflow.keras.utils import to_categorical

data_path = Path('../data')
IMAGE_SIZE = (50, 50)

print("Loading images...")
print("Note: Adjust paths based on your data structure")

# Example loading function
def load_images(image_paths, labels, target_size=IMAGE_SIZE):
    """Load and preprocess images."""
    images = []
    valid_labels = []
    for path, label in zip(image_paths, labels):
        try:
            img = cv2.imread(str(path))
            if img is not None:
                img = cv2.resize(img, target_size)
                images.append(img)
                valid_labels.append(label)
        except Exception as e:
            continue
    return np.array(images) / 255.0, np.array(valid_labels)

# Load data
# NOTE: Adjust paths and column names to match your dataset structure
metadata_file = data_path / 'metadata.csv'

if metadata_file.exists():
    df = pd.read_csv(metadata_file)
    print(f"✓ Metadata loaded: {len(df)} samples")
    
    # Adjust column names if your CSV uses different names
    # Expected columns: 'path' (image path) and 'label' (0 or 1)
    if 'path' in df.columns and 'label' in df.columns:
        X, y = load_images(df['path'].values, df['label'].values)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )
        Y_train = to_categorical(y_train, num_classes=2)
        Y_test = to_categorical(y_test, num_classes=2)
        print(f"✓ Images loaded: {len(X)} images")
        print(f"✓ Train: {len(X_train)}, Test: {len(X_test)}")
    else:
        print("⚠️ CSV missing required columns 'path' and 'label'")
        print("   Please adjust column names in the code above")
        X_train = X_test = Y_train = Y_test = None
else:
    print(f"⚠️ Metadata file not found: {metadata_file}")
    print("   Please:")
    print("   1. Create a CSV with columns: 'path', 'label'")
    print("   2. Place it in ../data/ folder")
    print("   3. Update metadata_file path if needed")
    print("   4. Re-run this cell")
    X_train = X_test = Y_train = Y_test = None

print(f"\nImage size: {IMAGE_SIZE}")
if X_train is not None:
    print("✓ Data ready for training")
else:
    print("⚠️ Data not loaded - fix paths above before training")

## 2. Data Generators

In [ ]:
# Data augmentation configuration
# Note: For this project, we load images directly (not using generators)
# But augmentation can be applied during training

BATCH_SIZE = 35
NUM_CLASSES = 2

print(f'Image size: {IMAGE_SIZE}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Number of classes: {NUM_CLASSES}')
print('\nNote: Images are loaded directly into memory for this small patch size')

## 3. Model Architecture

In [ ]:
from model import build_cnn_model

# Build model
model = build_cnn_model(input_shape=(*IMAGE_SIZE, 3))

print('Model architecture:')
model.summary()
print(f"\nTotal parameters: {model.count_params():,}")

## 4. Training

In [ ]:
# Model is already compiled in build_cnn_model
# But we can recompile if needed:
# model.compile(Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# Setup callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, verbose=1),
    ModelCheckpoint('../models/histopathology_model.h5', monitor='val_loss', 
                   save_best_only=True, verbose=1)
]

# Train model
EPOCHS = 40

# Uncomment when data is ready:
# history = model.fit(
#     X_train, Y_train,
#     validation_data=(X_test, Y_test),
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     callbacks=callbacks,
#     verbose=1
# )

print("Training configuration ready")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")

## Training History Visualization


In [ ]:
# Plot training history
if history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Loss
    axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[0].set_title('Model Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No training history available - train the model first")


## 5. Evaluation

In [ ]:
# Evaluate model
if X_test is not None and Y_test is not None:
    print("Evaluating model on test set...")
    test_results = model.evaluate(X_test, Y_test, verbose=1)
    print(f"\n✓ Test Loss: {test_results[0]:.4f}")
    print(f"✓ Test Accuracy: {test_results[1]:.4f}")
    
    # Make predictions
    Y_pred = model.predict(X_test, verbose=0)
    Y_pred_classes = np.argmax(Y_pred, axis=1)
    Y_true = np.argmax(Y_test, axis=1)
    
    # Classification report
    print("\n" + "="*60)
    print("Classification Report:")
    print("="*60)
    print(classification_report(Y_true, Y_pred_classes, 
                                target_names=['No Cancer', 'IDC Positive']))
    
    # Confusion matrix
    cm = confusion_matrix(Y_true, Y_pred_classes)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Cancer', 'IDC Positive'],
                yticklabels=['No Cancer', 'IDC Positive'])
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Accuracy
    from sklearn.metrics import accuracy_score
    accuracy = accuracy_score(Y_true, Y_pred_classes)
    print(f"\n✓ Overall Accuracy: {accuracy:.4f}")
else:
    print("⚠️ Cannot evaluate - data not loaded")
    print("   Please load data in the 'Data Preparation' cell above")